In [15]:
from ast import literal_eval
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import roc_auc_score
from tqdm import tqdm 

print("imports complete")

imports complete


In [9]:
from pathlib import Path

MODEL_NAME = "ibm-granite/granite-4.2-8b"
SHORT_MODEL_NAME = MODEL_NAME.split("/")[1]

DATA_DIR = Path("../../datasets")

CONDITION_DICT = {
    "no-prompt": DATA_DIR / "plain_dataset",
    "cot-zero-shot": DATA_DIR / "CoT_datasets" / "lexically_cleaned",
    "sentence-based-CoT": DATA_DIR / "CoT_datasets" / "sentence_based_lexically_cleaned",
    "no-prompt-chat-template": DATA_DIR / "plain_template_dataset",
    "ablation-filler-token": DATA_DIR / "ablation_datasets" / "filler_token_only",
    "ablation-instructions-only": DATA_DIR / "ablation_datasets" / "instructions_only",
}

csv_path = Path(f"../results_database.csv")

condition = "sentence-based-CoT" # Enter the condition you want to extract -- must be a key in CONDITION_DICT above, e.g. "no-prompt", "cot-zero-shot", "sentence-based-CoT", "no-prompt-chat-template", "ablation-filler-token", "ablation-instructions-only"

In [8]:
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()

print(f"✓ Model loaded")
print(f"  Device: {model.device}")
print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"  Layers: {model.config.num_hidden_layers}")

Loading tokenizer...


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

✓ Model loaded
  Device: cuda:0
  GPU memory: 17.58 GB
  Layers: 40


In [10]:
print(f"Loading {condition} datasets from {CONDITION_DICT[condition]}...")

# Load all task datasets
F0_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F0_train.csv")
F0_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F0_test.csv")
F1_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F1_train.csv")
F1_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F1_test.csv")
F2_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F2_train.csv")
F2_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F2_test.csv")
F3_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F3_train.csv")
F3_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F3_test.csv")
F4_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F4_train.csv")
F4_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F4_test.csv")
F5_train = pd.read_csv(f"{CONDITION_DICT[condition]}/F5_train.csv")
F5_test = pd.read_csv(f"{CONDITION_DICT[condition]}/F5_test.csv")
A1_train = pd.read_csv(f"{CONDITION_DICT[condition]}/A1_train.csv")
A1_test = pd.read_csv(f"{CONDITION_DICT[condition]}/A1_test.csv")
A2_train = pd.read_csv(f"{CONDITION_DICT[condition]}/A2_train.csv")
A2_test = pd.read_csv(f"{CONDITION_DICT[condition]}/A2_test.csv")
A3_train = pd.read_csv(f"{CONDITION_DICT[condition]}/A3_train.csv")
A3_test = pd.read_csv(f"{CONDITION_DICT[condition]}/A3_test.csv")

tasks_dict = {
    "F0": (F0_train, F0_test), "F1": (F1_train, F1_test), "F2": (F2_train, F2_test),
    "F3": (F3_train, F3_test), "F4": (F4_train, F4_test), "F5": (F5_train, F5_test),
    "A1": (A1_train, A1_test), "A2": (A2_train, A2_test), "A3": (A3_train, A3_test),
}
task_names = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]

print(f"✓ All {condition} datasets loaded:")
for name, (tr, te) in tasks_dict.items():
    print(f"    {name}: {len(tr)} train / {len(te)} test")

Loading sentence-based-CoT datasets from ..\..\datasets\CoT_datasets\sentence_based_lexically_cleaned...
✓ All sentence-based-CoT datasets loaded:
    F0: 999 train / 464 test
    F1: 1100 train / 491 test
    F2: 1033 train / 464 test
    F3: 1008 train / 478 test
    F4: 1135 train / 568 test
    F5: 1216 train / 562 test
    A1: 675 train / 297 test
    A2: 673 train / 286 test
    A3: 692 train / 300 test


In [13]:
def decode_ids(id_column):
    return [
        tokenizer.decode(literal_eval(ids) if isinstance(ids, str) else ids)
        for ids in id_column
    ]


def activations_all_layers(model, statements, batch_size=4):
    statements = list(statements)
    num_layers = model.config.num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_layers)]

    for i in tqdm(range(0, len(statements), batch_size)):
        batch = statements[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :]
            activations_by_layer[layer_idx].append(final_token.cpu())

        del outputs, inputs
        torch.cuda.empty_cache()

    return [torch.cat(layer_acts, dim=0) for layer_acts in activations_by_layer]


def train_probe(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels
    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(1000):
        optimizer.zero_grad()
        loss = loss_fn(probe(X_train_t).squeeze(-1), y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()

    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc


def train_all_layers(train_acts, test_acts, y_train, y_test):
    num_layers = len(train_acts)
    layer_results = {}
    for layer_idx in range(num_layers):
        X_train = train_acts[layer_idx]
        X_test = test_acts[layer_idx]
        X_train_np = torch.stack(list(X_train)).float().numpy()
        variance = X_train_np.var(axis=0) + 1e-6
        w, train_mean, auroc = train_probe((X_train, X_test), (y_train, y_test))
        layer_results[layer_idx] = {
            "auroc": auroc,
            "weights": w,
            "variance": variance,
            "train_mean": train_mean,
        }
        if layer_idx % 8 == 0:
            print(f"    Layer {layer_idx}: AUROC = {auroc:.4f}")
    return layer_results


def write_indomain_rows(task_name, layer_results, csv_path, condition, model_name):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        mask = ~((df["train_task"] == task_name) & 
                 (df["test_task"] == task_name) & 
                 (df["train_condition"] == condition) &
                 (df["model"] == model_name))
        df = df[mask]
    else:
        df = pd.DataFrame(columns=["train_task","test_task","train_condition","test_condition","layer","model","auroc"])

    new_rows = []
    for layer_idx, data in layer_results.items():
        new_rows.append({
            "train_task": task_name,
            "test_task": task_name,
            "train_condition": condition,
            "test_condition": condition,
            "layer": layer_idx,
            "model": model_name,
            "auroc": data["auroc"],
        })

    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"  ✓ Saved {len(new_rows)} layers for {task_name} ({condition}, {model_name})")

print("✓ All functions defined")

✓ All functions defined


In [16]:

task_name = "A1"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A1 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A1, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing A1...


100%|██████████| 75/75 [00:31<00:00,  2.38it/s]


    Layer 0: AUROC = 0.5374
    Layer 8: AUROC = 0.9747
    Layer 16: AUROC = 0.9922
    Layer 24: AUROC = 0.9941
    Layer 32: AUROC = 0.9950
    Layer 40: AUROC = 0.9968
  ✓ Saved 41 layers for A1 (sentence-based-CoT, granite-4.2-8b)


In [17]:

task_name = "A2"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A2 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A2, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing A2...


100%|██████████| 72/72 [00:25<00:00,  2.85it/s]


    Layer 0: AUROC = 0.5192
    Layer 8: AUROC = 0.9665
    Layer 16: AUROC = 0.9928
    Layer 24: AUROC = 0.9999
    Layer 32: AUROC = 1.0000
    Layer 40: AUROC = 0.9995
  ✓ Saved 41 layers for A2 (sentence-based-CoT, granite-4.2-8b)


In [18]:

task_name = "A3"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A3 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A3, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing A3...


100%|██████████| 75/75 [00:27<00:00,  2.74it/s]


    Layer 0: AUROC = 0.5567
    Layer 8: AUROC = 0.9636
    Layer 16: AUROC = 0.9933
    Layer 24: AUROC = 0.9997
    Layer 32: AUROC = 0.9997
    Layer 40: AUROC = 1.0000
  ✓ Saved 41 layers for A3 (sentence-based-CoT, granite-4.2-8b)


In [19]:
task_name = "F0"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F0 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F0, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F0...


100%|██████████| 116/116 [00:46<00:00,  2.48it/s]


    Layer 0: AUROC = 0.5551
    Layer 8: AUROC = 0.8138
    Layer 16: AUROC = 0.8958
    Layer 24: AUROC = 0.9380
    Layer 32: AUROC = 0.9771
    Layer 40: AUROC = 0.9750
  ✓ Saved 41 layers for F0 (sentence-based-CoT, granite-4.2-8b)


In [20]:

task_name = "F1"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F1 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F1, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F1...


100%|██████████| 123/123 [00:46<00:00,  2.63it/s]


    Layer 0: AUROC = 0.5033
    Layer 8: AUROC = 0.8382
    Layer 16: AUROC = 0.8997
    Layer 24: AUROC = 0.9624
    Layer 32: AUROC = 0.9843
    Layer 40: AUROC = 0.9855
  ✓ Saved 41 layers for F1 (sentence-based-CoT, granite-4.2-8b)


In [21]:

task_name = "F2"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F2 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F2, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F2...


100%|██████████| 116/116 [00:37<00:00,  3.06it/s]


    Layer 0: AUROC = 0.5720
    Layer 8: AUROC = 0.8281
    Layer 16: AUROC = 0.8984
    Layer 24: AUROC = 0.9597
    Layer 32: AUROC = 0.9845
    Layer 40: AUROC = 0.9834
  ✓ Saved 41 layers for F2 (sentence-based-CoT, granite-4.2-8b)


In [22]:

task_name = "F3"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F3 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F3, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F3...


100%|██████████| 120/120 [00:45<00:00,  2.63it/s]


    Layer 0: AUROC = 0.5416
    Layer 8: AUROC = 0.7673
    Layer 16: AUROC = 0.8627
    Layer 24: AUROC = 0.9397
    Layer 32: AUROC = 0.9610
    Layer 40: AUROC = 0.9500
  ✓ Saved 41 layers for F3 (sentence-based-CoT, granite-4.2-8b)


In [23]:

task_name = "F4"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F4 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F4, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F4...


100%|██████████| 142/142 [01:04<00:00,  2.21it/s]


    Layer 0: AUROC = 0.5199
    Layer 8: AUROC = 0.8119
    Layer 16: AUROC = 0.8572
    Layer 24: AUROC = 0.9525
    Layer 32: AUROC = 0.9738
    Layer 40: AUROC = 0.9699
  ✓ Saved 41 layers for F4 (sentence-based-CoT, granite-4.2-8b)


In [24]:

task_name = "F5"
print(f"\nProcessing {task_name}...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F5 = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F5, csv_path, condition=condition, model_name=SHORT_MODEL_NAME)


Processing F5...


100%|██████████| 141/141 [01:12<00:00,  1.94it/s]


    Layer 0: AUROC = 0.5695
    Layer 8: AUROC = 0.8542
    Layer 16: AUROC = 0.8994
    Layer 24: AUROC = 0.9633
    Layer 32: AUROC = 0.9790
    Layer 40: AUROC = 0.9678
  ✓ Saved 41 layers for F5 (sentence-based-CoT, granite-4.2-8b)


In [25]:
print("Pre-extracting FULL (train+test) activations for all tasks...")
full_activations = {}
full_labels = {}

for task_name in task_names:
    print(f"  Extracting {task_name}...")
    train_df, test_df = tasks_dict[task_name]
    full_df = pd.concat([train_df, test_df], ignore_index=True)  # use everything for out-of-domain testing
    full_labels[task_name] = full_df["label"].to_numpy()

    acts = activations_all_layers(model, decode_ids(full_df["extracted_statement_ids"]), batch_size=4)
    full_activations[task_name] = [
        layer_acts.float().numpy() if isinstance(layer_acts, torch.Tensor)
        else torch.stack(list(layer_acts)).float().numpy()
        for layer_acts in acts
    ]
    del acts
    torch.cuda.empty_cache()

print("\n All full activations extracted")


Pre-extracting FULL (train+test) activations for all tasks...
  Extracting A1...


100%|██████████| 243/243 [01:29<00:00,  2.73it/s]


  Extracting A2...


100%|██████████| 240/240 [01:17<00:00,  3.11it/s]


  Extracting A3...


...

  Extracting F4...


100%|██████████| 426/426 [03:07<00:00,  2.28it/s]


  Extracting F5...


100%|██████████| 445/445 [03:47<00:00,  1.96it/s]



 All full activations extracted


In [26]:
all_probes = {
    "A1": results_A1, "A2": results_A2, "A3": results_A3,
    "F0": results_F0, "F1": results_F1, "F2": results_F2,
    "F3": results_F3, "F4": results_F4, "F5": results_F5,
}

df = pd.read_csv(csv_path)
mask = ~(
    (df["train_task"] != df["test_task"]) &
    (df["model"] == SHORT_MODEL_NAME) &
    (df["train_condition"] == condition)
)
df = df[mask]

num_layers = model.config.num_hidden_layers + 1
cross_task_rows = []
total = len(task_names) * (len(task_names) - 1) * num_layers
completed = 0

for train_task in task_names:
    for test_task in task_names:
        if train_task == test_task:
            continue

        # Out-of-domain: evaluate on the FULL test_task data (train+test combined), not
        # just its held-out test split -- there's no train/test leakage concern here since
        # the probe was never trained on test_task at all.
        test_labels = full_labels[test_task]

        for layer_idx in range(num_layers):
            w = all_probes[train_task][layer_idx]["weights"]
            mean_train = all_probes[train_task][layer_idx]["train_mean"]

            X_test = full_activations[test_task][layer_idx]
            X_test_centered = X_test - mean_train
            logits = X_test_centered @ w
            auroc = roc_auc_score(test_labels, logits)

            cross_task_rows.append({
                "train_task": train_task,
                "test_task": test_task,
                "train_condition": condition,
                "test_condition": condition,
                "layer": layer_idx,
                "model": SHORT_MODEL_NAME,
                "auroc": auroc,
            })

            completed += 1
            if completed % 500 == 0:
                print(f"  {completed}/{total} done...")

df = pd.concat([df, pd.DataFrame(cross_task_rows)], ignore_index=True)
df.to_csv(csv_path, index=False)

print(f"\n✓ Done. Total rows: {len(df)}")
print(f"  In-domain: {len(df[df['train_task'] == df['test_task']])}")
print(f"  Cross-task: {len(df[df['train_task'] != df['test_task']])}")


  500/2952 done...
  1000/2952 done...
  1500/2952 done...
  2000/2952 done...
  2500/2952 done...

✓ Done. Total rows: 14013
  In-domain: 1557
  Cross-task: 12456


In [27]:
import pandas as pd

df = pd.read_csv(csv_path)

print("=" * 60)
print("CSV VALIDATION")
print("=" * 60)

print(f"\nTotal rows: {len(df)}")

print(f"\nRows by condition:")
conditions = df.groupby('train_condition').size()
print(conditions)

print(f"\nRows by model:")
models = df.groupby('model').size()
print(models)

print(f"\nIn-domain rows (train_task == test_task):")
indomain = df[df['train_task'] == df['test_task']]
print(f"Total: {len(indomain)}")
print(indomain.groupby('train_condition').size())

print(f"\nCross-task rows (train_task != test_task):")
crosstask = df[df['train_task'] != df['test_task']]
print(f"Total: {len(crosstask)}")
print(crosstask.groupby('train_condition').size())

print(f"\nLayer 25 AUROC by condition:")
layer25 = df[df['layer'] == 25].pivot_table(
    index='train_task',
    columns='train_condition',
    values='auroc'
)
print(layer25.round(4))

print(f"\n✓ Validation complete")

CSV VALIDATION

Total rows: 14013

Rows by condition:
train_condition
cot-zero-shot              2673
no-prompt                  2673
no-prompt-chat-template    2673
sentence-based-CoT         5994
dtype: int64

Rows by model:
model
deepseek-r1-distill-8b    10692
granite-4.2-8b             3321
dtype: int64

In-domain rows (train_task == test_task):
Total: 1557
train_condition
cot-zero-shot              297
no-prompt                  297
no-prompt-chat-template    297
sentence-based-CoT         666
dtype: int64

Cross-task rows (train_task != test_task):
Total: 12456
train_condition
cot-zero-shot              2376
no-prompt                  2376
no-prompt-chat-template    2376
sentence-based-CoT         5328
dtype: int64

Layer 25 AUROC by condition:
train_condition  cot-zero-shot  no-prompt  no-prompt-chat-template  \
train_task                                                           
A1                      0.9443     0.5967                   0.7216   
A2                      0.93